# Transfer Learning for Computer Vision

Leverage pre-trained models for custom image classification tasks.

## Learning Objectives

- Understand transfer learning concepts
- Fine-tune pre-trained models for custom datasets
- Implement feature extraction vs full fine-tuning
- Apply best practices for efficient training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import copy

import torchvision
from torchvision import datasets, models
from torchvision.transforms import v2
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Transfer Learning Concepts

Transfer learning reuses knowledge from one task for another:

```
Pre-trained Model (ImageNet)    Your Task
┌──────────────────────┐        ┌──────────────────────┐
│ Conv Layers (frozen) │   →    │ Conv Layers (frozen) │
│ - Edge detectors     │        │ - Same features      │
│ - Texture patterns   │        │                      │
│ - Object parts       │        │                      │
├──────────────────────┤        ├──────────────────────┤
│ FC: 1000 classes     │   →    │ FC: N classes        │
│ (replaced)           │        │ (new classifier)     │
└──────────────────────┘        └──────────────────────┘
```

### Two Strategies:
1. **Feature Extraction**: Freeze all layers, train only new classifier
2. **Fine-tuning**: Train all or later layers with low learning rate

In [ ]:
# Visualize what features are learned at different depths
feature_description = """
=== Features Learned at Different Network Depths ===

Early Layers (Transfer well to most tasks):
  • Edges and gradients
  • Color blobs
  • Basic textures

Middle Layers (Transfer to similar domains):
  • Texture patterns
  • Object parts (wheels, eyes, fur)
  • Shapes and contours

Later Layers (Task-specific):
  • Object categories
  • Scene layouts
  • Fine-grained distinctions

=== When to Use Which Strategy ===

┌─────────────────────────┬─────────────────────────────────────┐
│ Scenario                │ Strategy                            │
├─────────────────────────┼─────────────────────────────────────┤
│ Small dataset (<1000)   │ Feature extraction only            │
│ Medium dataset (1k-10k) │ Fine-tune later layers             │
│ Large dataset (>10k)    │ Fine-tune entire network           │
│ Similar to ImageNet     │ Less training needed               │
│ Very different domain   │ More training, possibly from layer │
└─────────────────────────┴─────────────────────────────────────┘
"""
print(feature_description)

## 2. Prepare Dataset

We'll use CIFAR-10 as our target dataset with a subset for demonstration.

In [ ]:
# Define transforms
# Training: augmentation + normalization
train_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224), antialias=True),  # ResNet expects 224x224
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Validation: only resize + normalization
val_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load CIFAR-10
train_dataset = datasets.CIFAR10(
    root='./datasets', train=True, download=True, transform=train_transforms
)
val_dataset = datasets.CIFAR10(
    root='./datasets', train=False, download=True, transform=val_transforms
)

# Use subset for faster training (demo purposes)
train_subset = Subset(train_dataset, range(5000))  # 5000 training samples
val_subset = Subset(val_dataset, range(1000))      # 1000 validation samples

print(f"Training samples: {len(train_subset)}")
print(f"Validation samples: {len(val_subset)}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
# Create DataLoaders
batch_size = 32

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0)

# Visualize some samples
def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(train_dataset.classes[labels[i]], fontsize=10)
    ax.axis('off')
plt.suptitle('Training Samples (with augmentation)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Strategy 1: Feature Extraction

Freeze all pre-trained layers, only train the new classifier head.

In [ ]:
def create_feature_extractor(num_classes=10):
    """Create a ResNet18 feature extractor."""
    # Load pre-trained model
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    
    # Freeze all parameters
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace classifier head
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    
    return model


# Create model
feature_model = create_feature_extractor(num_classes=10).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in feature_model.parameters())
trainable_params = sum(p.numel() for p in feature_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"\nTrainable ratio: {trainable_params/total_params:.2%}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100.0 * correct / total

In [ ]:
# Training settings for feature extraction
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(feature_model.fc.parameters(), lr=0.001)  # Only train FC layer
num_epochs = 5

# Training loop
print("Training with Feature Extraction...")
feature_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(feature_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(feature_model, val_loader, criterion, device)
    
    feature_history['train_loss'].append(train_loss)
    feature_history['train_acc'].append(train_acc)
    feature_history['val_loss'].append(val_loss)
    feature_history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

print(f"\nFinal Validation Accuracy: {val_acc:.2f}%")

## 4. Strategy 2: Fine-Tuning

Unfreeze some or all layers and train with a lower learning rate.

In [ ]:
def create_finetune_model(num_classes=10, freeze_until='layer3'):
    """
    Create a ResNet18 for fine-tuning.
    
    freeze_until: Freeze all layers before this (None = train all)
                  Options: 'layer1', 'layer2', 'layer3', 'layer4', None
    """
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    
    # Layer names in order
    layer_names = ['conv1', 'bn1', 'layer1', 'layer2', 'layer3', 'layer4']
    
    if freeze_until is not None:
        freeze_idx = layer_names.index(freeze_until) + 1
        
        # Freeze early layers
        for name, param in model.named_parameters():
            layer = name.split('.')[0]
            if layer in layer_names[:freeze_idx]:
                param.requires_grad = False
    
    # Replace classifier
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    
    return model


# Create fine-tuning model (freeze until layer3)
finetune_model = create_finetune_model(num_classes=10, freeze_until='layer2').to(device)

# Count parameters
total_params = sum(p.numel() for p in finetune_model.parameters())
trainable_params = sum(p.numel() for p in finetune_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable ratio: {trainable_params/total_params:.2%}")

In [ ]:
# Use different learning rates for different layers
def get_layer_groups(model):
    """Group parameters by layer for differential learning rates."""
    groups = [
        {'params': model.layer3.parameters(), 'lr': 1e-4},  # Lower LR for pretrained
        {'params': model.layer4.parameters(), 'lr': 5e-4},  # Medium LR
        {'params': model.fc.parameters(), 'lr': 1e-3},      # Higher LR for new layers
    ]
    return groups

# Setup optimizer with differential learning rates
param_groups = get_layer_groups(finetune_model)
optimizer = optim.AdamW(param_groups, weight_decay=0.01)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print("Optimizer parameter groups:")
for i, group in enumerate(optimizer.param_groups):
    print(f"  Group {i}: LR={group['lr']}")

In [ ]:
# Training loop for fine-tuning
print("Training with Fine-Tuning...")
finetune_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_acc = 0.0
best_model_weights = None

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(finetune_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(finetune_model, val_loader, criterion, device)
    scheduler.step()
    
    finetune_history['train_loss'].append(train_loss)
    finetune_history['train_acc'].append(train_acc)
    finetune_history['val_loss'].append(val_loss)
    finetune_history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_weights = copy.deepcopy(finetune_model.state_dict())
    
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

print(f"\nBest Validation Accuracy: {best_acc:.2f}%")

In [ ]:
# Compare training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, num_epochs + 1)

# Loss comparison
axes[0].plot(epochs, feature_history['train_loss'], 'b--', label='Feature Ext. (Train)')
axes[0].plot(epochs, feature_history['val_loss'], 'b-', label='Feature Ext. (Val)')
axes[0].plot(epochs, finetune_history['train_loss'], 'r--', label='Fine-Tune (Train)')
axes[0].plot(epochs, finetune_history['val_loss'], 'r-', label='Fine-Tune (Val)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()

# Accuracy comparison
axes[1].plot(epochs, feature_history['train_acc'], 'b--', label='Feature Ext. (Train)')
axes[1].plot(epochs, feature_history['val_acc'], 'b-', label='Feature Ext. (Val)')
axes[1].plot(epochs, finetune_history['train_acc'], 'r--', label='Fine-Tune (Train)')
axes[1].plot(epochs, finetune_history['val_acc'], 'r-', label='Fine-Tune (Val)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Accuracy Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Using EfficientNet

EfficientNet often provides better accuracy/efficiency tradeoff.

In [ ]:
def create_efficientnet_classifier(num_classes=10, freeze_backbone=True):
    """Create an EfficientNet-B0 classifier."""
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    
    if freeze_backbone:
        # Freeze feature extractor
        for param in model.features.parameters():
            param.requires_grad = False
    
    # Replace classifier
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(num_features, num_classes)
    )
    
    return model


# Create and analyze
efficient_model = create_efficientnet_classifier(num_classes=10)

total = sum(p.numel() for p in efficient_model.parameters())
trainable = sum(p.numel() for p in efficient_model.parameters() if p.requires_grad)

print(f"EfficientNet-B0 for CIFAR-10:")
print(f"  Total parameters: {total:,}")
print(f"  Trainable parameters: {trainable:,}")

## 6. Gradual Unfreezing

Progressively unfreeze layers during training for better results.

In [ ]:
def gradual_unfreeze_resnet(model, stage):
    """
    Gradually unfreeze ResNet layers.
    
    Stages:
        0: Only classifier trainable
        1: + layer4 trainable
        2: + layer3 trainable
        3: + layer2 trainable
        4: All trainable
    """
    # First freeze everything
    for param in model.parameters():
        param.requires_grad = False
    
    # Always train classifier
    for param in model.fc.parameters():
        param.requires_grad = True
    
    # Unfreeze based on stage
    layers_to_unfreeze = [model.layer4, model.layer3, model.layer2, model.layer1]
    
    for i in range(min(stage, len(layers_to_unfreeze))):
        for param in layers_to_unfreeze[i].parameters():
            param.requires_grad = True
    
    # Count trainable
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable


# Demonstrate gradual unfreezing
demo_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
demo_model.fc = nn.Linear(512, 10)

print("Gradual Unfreezing Stages:")
for stage in range(5):
    trainable = gradual_unfreeze_resnet(demo_model, stage)
    print(f"  Stage {stage}: {trainable:,} trainable parameters")

## 7. Learning Rate Finder

Find optimal learning rate for fine-tuning.

In [ ]:
def lr_finder(model, train_loader, criterion, device, start_lr=1e-7, end_lr=10, num_iters=100):
    """Find optimal learning rate using LR range test."""
    model = copy.deepcopy(model).to(device)
    optimizer = optim.SGD(model.parameters(), lr=start_lr)
    
    # Exponential LR schedule
    lr_mult = (end_lr / start_lr) ** (1 / num_iters)
    
    lrs = []
    losses = []
    best_loss = float('inf')
    
    model.train()
    iterator = iter(train_loader)
    
    for i in range(num_iters):
        try:
            images, labels = next(iterator)
        except StopIteration:
            iterator = iter(train_loader)
            images, labels = next(iterator)
        
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Stop if loss explodes
        if loss.item() > 4 * best_loss:
            break
        
        best_loss = min(best_loss, loss.item())
        
        loss.backward()
        optimizer.step()
        
        lrs.append(optimizer.param_groups[0]['lr'])
        losses.append(loss.item())
        
        # Update learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] *= lr_mult
    
    return lrs, losses


# Run LR finder
print("Running learning rate finder...")
test_model = create_finetune_model(num_classes=10, freeze_until=None)
lrs, losses = lr_finder(test_model, train_loader, criterion, device)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(lrs, losses)
plt.xscale('log')
plt.xlabel('Learning Rate (log scale)')
plt.ylabel('Loss')
plt.title('Learning Rate Finder')
plt.grid(True)

# Find suggested LR (steepest descent)
min_idx = np.argmin(losses)
suggested_lr = lrs[min_idx] / 10  # Use LR 10x before minimum
plt.axvline(x=suggested_lr, color='r', linestyle='--', label=f'Suggested: {suggested_lr:.1e}')
plt.legend()
plt.show()

print(f"Suggested learning rate: {suggested_lr:.1e}")

## 8. Best Practices Summary

In [ ]:
best_practices = """
=== Transfer Learning Best Practices ===

1. DATA PREPARATION
   ✓ Use same normalization as pre-training (ImageNet stats)
   ✓ Resize to expected input size (224x224 for most models)
   ✓ Apply augmentation only to training data

2. MODEL SETUP
   ✓ Start with frozen backbone for small datasets
   ✓ Replace classifier head with appropriate output size
   ✓ Consider gradual unfreezing for better results

3. TRAINING
   ✓ Use lower learning rate for pre-trained layers (10-100x lower)
   ✓ Use weight decay for regularization (0.01 typical)
   ✓ Use learning rate scheduling (cosine, step decay)
   ✓ Early stopping with patience

4. OPTIMIZATION
   ✓ AdamW often works better than SGD for fine-tuning
   ✓ Use LR finder to find optimal learning rate
   ✓ Warm up learning rate for first few epochs

5. REGULARIZATION
   ✓ Dropout in classifier head
   ✓ Data augmentation
   ✓ Label smoothing (optional)
   ✓ Mixup/CutMix for larger datasets

6. EVALUATION
   ✓ Use validation set for hyperparameter tuning
   ✓ Keep test set separate for final evaluation
   ✓ Report confidence intervals if possible
"""
print(best_practices)

## Summary

### Key Concepts

| Strategy | When to Use | Learning Rate |
|----------|-------------|---------------|
| Feature Extraction | Small dataset, similar domain | Higher (1e-3) |
| Fine-tune (partial) | Medium dataset | Medium (1e-4 to 1e-3) |
| Fine-tune (full) | Large dataset | Lower (1e-5 to 1e-4) |

### Results Comparison

| Method | Final Val Accuracy | Trainable Params |
|--------|-------------------|------------------|
| Feature Extraction | ~70-75% | ~135K |
| Fine-tuning | ~80-85% | ~5-11M |

In [ ]:
# Final comparison
print("=== Final Results ===")
print(f"\nFeature Extraction: {feature_history['val_acc'][-1]:.2f}% validation accuracy")
print(f"Fine-tuning: {finetune_history['val_acc'][-1]:.2f}% validation accuracy")
print("\nNotebook completed successfully!")